### 1. Տվյալների հավաքագրում (Scraping / Request) և Մաքրում

Այս բջջում իրականացվում են հետևյալ գործողությունները.
* **Հարցումների կատարում:** Տվյալների հավաքագրում կայքից կամ API-ից Selenium/Requests գրադարանների միջոցով։
* **Տվյալների մաքրում:** Ստացված հում տվյալների զտում, ավելորդ կամ դատարկ արժեքների հեռացում և կառուցվածքի բերում Pandas DataFrame ձևաչափի։
* **Պահպանում:** Մաքրված տվյալների տեղական գրանցում `products.parquet` ֆայլի տեսքով՝ հետագա միգրացիայի համար։
  

In [2]:
!pip install requests beautifulsoup4 pandas pyarrow

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [4]:
url = "https://www.zara.com/am/en/woman-dresses-l1066.html"

print(url)

https://www.zara.com/am/en/woman-dresses-l1066.html


In [5]:
response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

print("Status code:", response.status_code)
print("HTML size:", len(response.text))

Status code: 200
HTML size: 2240


In [6]:
raw_html = response.text

print(raw_html[:3000])

<!DOCTYPE html><html><head> <meta charset="utf-8"> <meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no"> <meta http-equiv="refresh" content="5; URL='/am/en/woman-dresses-l1066.html?bm-verify=AAQAAAAO_____wxWQPmRR0Al9Y5RYEsuMOXNNUIiYBtuqrorAKkeMpPSiOAP_upTf1s1gLLjsq8MvgBtDiLWhShxWxo2k6osgM9dA4K_r0UxBrUBK0WBKLZj8MR8DENOd76j2Dghid2zxcKKHWG5qjGzV-_ewz5p8o-ie05n4m0IEM7SRQRHiYAMczU3hRRl1Rmgnqiln1Pn-aPcP1lEzxnBzeV9BaVAis0jmIPlsnJWTCi0GzY-W_AUDtbreNHpBgIG6GM7tdT2wJhhao9DVxD8sxaca7AUdw'" /><title>&nbsp;</title><script> var i = 1792938150; var j = i + Number("4872" + "94963"); </script> </head> <noscript> <iframe style="border: none; height: 100%; width: 100%;" src=""></iframe></noscript><body> <iframe style="border: none; width: 100vw; height: 100vh;" src="/interstitial/ic.html"> </iframe> <script> function triggerInterstitialChallenge() {var xhr = new XMLHttpRequest(); xhr.withCredentials = true; xhr.addEventListener("loadend", function() { try { var data = JSO

In [8]:
!pip install selenium

   ---------------------------------------- 0.0/11.8 MB ? eta -:--:--
    --------------------------------------- 0.3/11.8 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.8 MB 4.5 MB/s eta 0:00:03
   -------- ------------------------------- 2.6/11.8 MB 5.5 MB/s eta 0:00:02
   ------------ --------------------------- 3.7/11.8 MB 5.6 MB/s eta 0:00:02
   ------------------ --------------------- 5.5/11.8 MB 6.0 MB/s eta 0:00:02
   --------------------- ------------------ 6.3/11.8 MB 6.2 MB/s eta 0:00:01
   ------------------------ --------------- 7.3/11.8 MB 5.7 MB/s eta 0:00:01
   ------------------------------ --------- 8.9/11.8 MB 5.9 MB/s eta 0:00:01
   ----------------------------------- ---- 10.5/11.8 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------  11.5/11.8 MB 6.1 MB/s eta 0:00:01
   ---------------------------------------- 11.8/11.8 MB 5.9 MB/s  0:00:02

   ---------------------------------------- 0/6 [wsproto]
   ----------------------------

In [9]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# Բրաուզերի կարգավորումները
options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

try:
    url = "https://www.zara.com/am/en/woman-dresses-l1066.html"
    print("Բացվում է Զարայի էջը...")
    driver.get(url)
    time.sleep(3)
    
    # Իջնում ենք ներքև մինչև 1000 ապրանք բեռնվի կամ էջի վերջը հասնի
    target_count = 1000
    last_height = driver.execute_script("return document.body.scrollHeight")
    
    print("Սկսվում է ապրանքների բեռնման գործընթացը (scroll)...")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2.5)
        
        current_elements = driver.find_elements(By.CSS_SELECTOR, "a.product-link")
        print(f"Առայժմ գտնվել է հղում: {len(current_elements)}")
        
        if len(current_elements) >= target_count:
            print("Հասանք նպատակային քանակին:")
            break
            
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            print("Էջի վերջն է, նոր ապրանքներ չեն ավելանում:")
            break
        last_height = new_height

    # Հավաքում ենք բոլոր եզակի հղումները
    product_elements = driver.find_elements(By.CSS_SELECTOR, "a.product-link")
    raw_links = [elem.get_attribute("href") for elem in product_elements if elem.get_attribute("href")]
    raw_links = list(set(raw_links))
    print(f"Ընդհանուր գտնվեց եզակի հղում: {len(raw_links)}")

finally:
    # Ժամանակավորապես թողնում ենք driver-ը բաց, որ հետո օգտագործենք մանրամասների համար
    pass

Բացվում է Զարայի էջը...
Սկսվում է ապրանքների բեռնման գործընթացը (scroll)...
Առայժմ գտնվել է հղում: 124
Առայժմ գտնվել է հղում: 192
Առայժմ գտնվել է հղում: 250
Առայժմ գտնվել է հղում: 310
Առայժմ գտնվել է հղում: 366
Առայժմ գտնվել է հղում: 420
Առայժմ գտնվել է հղում: 476
Առայժմ գտնվել է հղում: 533
Առայժմ գտնվել է հղում: 590
Առայժմ գտնվել է հղում: 646
Առայժմ գտնվել է հղում: 702
Առայժմ գտնվել է հղում: 756
Առայժմ գտնվել է հղում: 816
Առայժմ գտնվել է հղում: 880
Առայժմ գտնվել է հղում: 942
Առայժմ գտնվել է հղում: 1004
Հասանք նպատակային քանակին:
Ընդհանուր գտնվեց եզակի հղում: 449


In [13]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

products_data = []

try:
    url = "https://www.zara.com/am/en/woman-dresses-l1066.html"
    print("Բացվում է Զարայի էջը...")
    driver.get(url)
    time.sleep(3)
    
    # Արագ հավաքում ենք հղումները (մոտ 100-150 հատ բավական է)
    for _ in range(3):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        
    current_elements = driver.find_elements(By.CSS_SELECTOR, "a.product-link")
    raw_links = list(set([elem.get_attribute("href") for elem in current_elements if elem.get_attribute("href")]))
    
    # Սահմանափակում ենք օրինակ առաջին 50-ով, որ արագ աշխատի
    selected_links = raw_links[:50]
    print(f"Ընտրվեց {len(selected_links)} ապրանք արագ ստուգման և մշակման համար:")

    for idx, link in enumerate(selected_links):
        driver.get(link)
        time.sleep(1)
        
        try:
            product_name = driver.find_element(By.CSS_SELECTOR, "h1.product-detail-info__header-name").text.strip()
        except:
            product_name = None

        try:
            price_text = driver.find_element(By.CSS_SELECTOR, "span.money-amount__main").text.replace(" ", "").replace(",", "")
            clean_price = float(''.join(c for c in price_text if c.isdigit() or c == '.'))
        except:
            clean_price = None

        try:
            image_url = driver.find_element(By.CSS_SELECTOR, "img.media-image__image").get_attribute("src")
        except:
            image_url = None

        product_id = link.split("-p")[-1].split(".html")[0] if "-p" in link else str(idx + 1)

        products_data.append({
            "product_id": product_id,
            "product_name": product_name,
            "price": clean_price,
            "currency": "AMD",
            "product_url": link,
            "image_url": image_url,
            "category": "Women -> Dresses"
        })

finally:
    driver.quit()

# 1. Տվյալների ստեղծում և մաքրում (Data Cleaning)
df_raw = pd.DataFrame(products_data)
df_cleaned = df_raw.dropna(subset=['product_name', 'price', 'product_url']).copy()
df_cleaned = df_cleaned.drop_duplicates(subset=['product_url'])
df_cleaned['image_url'] = df_cleaned['image_url'].fillna('')

print(f"\nՄաքրումից հետո մնաց {len(df_cleaned)} վավեր ապրանք:")

# 2. Պահպանում Parquet ֆայլում
output_filename = "products.parquet"
df_cleaned.to_parquet(output_filename, index=False)
print(f"Տվյալները հաջողությամբ պահպանվեցին '{output_filename}' ֆայլում!")

# Ցուցադրում ենք արդյունքը
display(df_cleaned.head())

Բացվում է Զարայի էջը...
Ընտրվեց 50 ապրանք արագ ստուգման և մշակման համար:

Մաքրումից հետո մնաց 50 վավեր ապրանք:
Տվյալները հաջողությամբ պահպանվեցին 'products.parquet' ֆայլում!


,product_id,product_name,price,currency,product_url,image_url,category
0,04786307,ZW COLLECTION HALTER SATIN MIDI DRESS,25900.0,AMD,https://www.zara.com/am/en/zw-collection-halte...,,Women -> Dresses
1,07853332,SATIN OPENING MIDI DRESS,25900.0,AMD,https://www.zara.com/am/en/satin-midi-dress-wi...,,Women -> Dresses
2,01355455,CONTOURING MIDI DRESS,23900.0,AMD,https://www.zara.com/am/en/contouring-midi-dre...,,Women -> Dresses
3,05584801,STRETCH MINI DRESS WITH ZIPS,15900.0,AMD,https://www.zara.com/am/en/stretch-mini-dress-...,,Women -> Dresses
4,05039845,ANIMAL PRINT LONG SLIT DRESS,17900.0,AMD,https://www.zara.com/am/en/animal-print-slit-m...,,Women -> Dresses


In [14]:
print(f"Աղյուսակի իրական տողերի քանակը՝ {len(df_cleaned)}")
# Կարող եք տեսնել բոլոր սյունակների անունները
print(df_cleaned.columns.tolist())

Աղյուսակի իրական տողերի քանակը՝ 50
['product_id', 'product_name', 'price', 'currency', 'product_url', 'image_url', 'category']


In [15]:
# Պահպանում ենք մշակված տվյալները Parquet ֆայլի տեսքով
df_cleaned.to_parquet(r"C:\Users\asatr\Downloads\products.parquet", index=False)
print("Ֆայլը հաջողությամբ պահպանվեց Downloads պանակում:")

Ֆայլը հաջողությամբ պահպանվեց Downloads պանակում:


### 3. **Database Migration, Timestamping, and Persistence**

Այս փուլում իրականացվում է մշակված տվյալների պահպանումը և միգրացիան դեպի տեղական SQL Server բազա.
* **Parquet Persistence:** Մաքրված տվյալներիֆրեյմը արդյունավետորեն պահպանվում է տեղական `Parquet` ֆորմատով։
* **Dynamic Timestamping:** Հարցման պահին ավտոմատ կերպով ավելացվում է `request_date` սյունակը (ընթացիկ ամսաթվով և ժամով)՝ հարցումների պատմությունը պահպանելու նպատակով։
* **SQL Server Integration:** Օգտագործելով `SQLAlchemy` և `pyodbc` գործիքակազմը, տվյալները հաջողությամբ բեռնվում են `ZaraDB`-ի `Products` աղյուսակ՝ `append` ռեժիմով։

In [ ]:
from datetime import datetime
import pandas as pd
from sqlalchemy import create_engine

# 1. Կարդում ենք պահպանված Parquet ֆայլը Downloads պանակից
file_path = r"C:\Users\asatr\Downloads\products.parquet"
df = pd.read_parquet(file_path)

# 2. Ավելացնում ենք հարցման ընթացիկ ամսաթիվն ու ժամը
df["request_date"] = datetime.now()
print(
    f"Ֆայլից հաջողությամբ կարդացվեց {len(df)} ապրանք, և ավելացվեց request_date"
    " սյունակը։"
)

# 3. Կապ SQL Server-ի ZaraDB տվյալների բազայի հետ
conn_str = (
    "mssql+pyodbc://localhost/ZaraDB"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)
engine = create_engine(conn_str)

# 4. Բեռնում ենք տվյալները բազա (append ռեժիմով)
df.to_sql("Products", con=engine, if_exists="append", index=False)

print(
    "Տվյալները հաջողությամբ և անխափան գրանցվեցին ZaraDB-ի Products աղյուսակում։"
)